### Amazon Kindle Review Sentiment Analysis

##### About dataset:


Context :

This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content:

5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

In [1]:
#importing libraries

import numpy as np
import pandas as pd

import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from bs4 import BeautifulSoup


In [2]:
#importing dataset into dataframe

df = pd.read_csv("datasets\\all_kindle_review .csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
#discarding other columns

df = df[['rating', 'reviewText']]
df.head()

,rating,reviewText
0,3,"Jace Rankin may be short, but he's nothing to ..."
1,5,Great short read. I didn't want to put it dow...
2,3,I'll start by saying this is the first of four...
3,3,Aggie is Angela Lansbury who carries pocketboo...
4,4,I did not expect this type of book to be in li...


In [4]:
#checking for null values or duplicates

df.isnull().sum()

rating        0
reviewText    0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
#analyzing rating column

df['rating'].unique()

array([3, 5, 4, 2, 1])

In [7]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [8]:
#changing data in rating column to binary values
#rating less than 3 as 0 and greater than 3 as 1.

df['rating']=df['rating'].apply(lambda x: 0 if x < 3 else 1)

In [9]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [10]:
#text preprocessing on reviewText column

#converting into lower case

df['reviewText'] = df['reviewText'].str.lower()

In [11]:
df['reviewText'].head()

0    jace rankin may be short, but he's nothing to ...
1    great short read.  i didn't want to put it dow...
2    i'll start by saying this is the first of four...
3    aggie is angela lansbury who carries pocketboo...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: str

In [ ]:
#removing special chars
df['reviewText'] = df['reviewText'].apply(lambda x : re.sub('[^a-zA-Z0-9]', " ",x))

#removing stopwords from text
df['reviewText'] = df['reviewText'].apply(lambda x : " ".join([y for y in x.split(" ") if y not in set (stopwords.words("english"))]))

#removing url's from text
df['reviewText'] = df['reviewText'].apply(lambda x : re.sub(r"(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?", "", str(x)))

#removing html tags
df['reviewText'] = df['reviewText'].apply(lambda x : BeautifulSoup(x, "lxml").get_text())



In [ ]:
wordnet_lemmatizer = WordNetLemmatizer()

df['reviewText'] = " ".join([wordnet_lemmatizer.lemmatize(word) for word in df['reviewText'].str.split(" ")])


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(df['reviewText'], df['review'], test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vectorizer = CountVectorizer(binary=True, max_features=100,ngram_range=(1,1))
x_train_arr_cv = count_vectorizer.fit_transform(x_train).toarray()
x_test_arr_cv = count_vectorizer.transform(x_test).toarray()

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=100,ngram_range=(1,1))
x_train_arr_tfidf = tfidf_vectorizer.fit_transform(x_train).toarray()
x_test_arr_tfidf = tfidf_vectorizer.transform(x_test).toarray()

In [ ]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB()

gaussian_nb_bow = gaussian_nb.fit_transform(x_train_arr_cv, y_train)
gaussian_nb_tfidf = gaussian_nb.fit_transform(x_train_arr_tfidf, y_train)

In [ ]:
y_pred_bow = gaussian_nb_bow.predict(x_test_arr_cv)
y_pred_tfidf = gaussian_nb_tfidf.predict(x_test_arr_tfidf)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_bow= accuracy_score(y_test, y_pred_bow)
accuracy_tfidf= accuracy_score(y_test, y_pred_tfidf)